# Basic Engineering Mechanics: Symbolic Analysis of Stress, Strain, and Axial Loading

This notebook implements symbolic calculations for fundamental 1st-year mechanics of materials:
1. Hooke's Law, axial deformation, and Modulus of Resilience (area under elastic curve).
2. Deriving relations among Elastic Constants ($E$, $G$, $K$, $\nu$) using simultaneous equation solving.
3. Solving a Statically Indeterminate axially loaded stepped bar (Equilibrium + Compatibility).
4. Direct shear stress and Factor of Safety (FOS) evaluation for connecting pins.

In [1]:
import sympy as sym
from IPython.display import Math, Latex

### 1. Uniaxial Stress, Elongation, and Modulus of Resilience
For a uniform bar of length $L$, cross-sectional area $A$, and Young's modulus $E$ subjected to axial force $P$:
* Normal stress: $\sigma = \frac{P}{A}$
* Normal strain: $\epsilon = \frac{\sigma}{E}$
* Axial elongation: $\delta = \frac{PL}{AE}$

The **Modulus of Resilience** ($u_r$) is the strain energy stored per unit volume up to the elastic limit ($\epsilon_y$):
$$u_r = \int_0^{\epsilon_y} \sigma(\epsilon) \, d\epsilon = \int_0^{\epsilon_y} E \epsilon \, d\epsilon$$

In [2]:
P, L, A, E, epsilon = sym.symbols('P, L, A, E, epsilon', positive=True)
epsilon_y, sigma_y = sym.symbols('epsilon_y, sigma_y', positive=True)

# Stress and Elongation
sigma = P / A
delta = (P * L) / (A * E)

# Stress as a function of strain in the elastic region
stress_fn = E * epsilon

# Symbolic integration for Modulus of Resilience
u_r = sym.integrate(stress_fn, (epsilon, 0, epsilon_y))

# Substitute epsilon_y = sigma_y / E
u_r_simplified = u_r.subs(epsilon_y, sigma_y / E)

display(Math(r'\sigma = {}'.format(sym.latex(sigma))))
display(Math(r'\delta = {}'.format(sym.latex(delta))))
display(Math(r'u_r = \int_0^{{\epsilon_y}} E\epsilon\,d\epsilon = {} = {}'.format(
    sym.latex(u_r), sym.latex(u_r_simplified)
)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 2. Relations Between Elastic Constants ($E$, $G$, $K$, $\nu$)
From generalized Hooke's law, the elastic constants are related by:
1. $E = 2G(1 + \nu)$
2. $E = 3K(1 - 2\nu)$

We set up these two relations as symbolic equations using `sym.Eq` and solve for $E$ purely in terms of Bulk Modulus ($K$) and Shear Modulus ($G$).

In [3]:
E_sym, G_sym, K_sym, nu_sym = sym.symbols('E, G, K, nu', positive=True)

eq_G = sym.Eq(E_sym, 2 * G_sym * (1 + nu_sym))
eq_K = sym.Eq(E_sym, 3 * K_sym * (1 - 2 * nu_sym))

display(eq_G, eq_K)

# Solve for Poisson's ratio (nu) and Young's modulus (E) in terms of G and K
soln = sym.solve([eq_G, eq_K], (nu_sym, E_sym))

display(Math(r'\nu = {}'.format(sym.latex(soln[nu_sym]))))
display(Math(r'E = \frac{{9KG}}{{3K + G}} = {}'.format(sym.latex(soln[E_sym]))))

Eq(E, 2*G*(nu + 1))

Eq(E, 3*K*(1 - 2*nu))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 3. Statically Indeterminate Axial Loading
Consider a stepped cylindrical steel bar fixed rigidly at both ends ($A$ and $C$). 
* Segment 1 (length $L_1$, area $A_1$, modulus $E$)
* Segment 2 (length $L_2$, area $A_2$, modulus $E$)
An axial load $P$ is applied at the junction $B$. 

Because the total elongation between rigid supports must be zero, we apply:
1. **Equilibrium:** $R_A + R_C = P$
2. **Compatibility:** $\delta_{AB} + \delta_{BC} = 0 \implies \frac{R_A L_1}{A_1 E} - \frac{R_C L_2}{A_2 E} = 0$

In [4]:
R_A, R_C, P_load = sym.symbols('R_A, R_C, P')
L_1, L_2, A_1, A_2 = sym.symbols('L_1, L_2, A_1, A_2', positive=True)

# 1. Equilibrium Equation: R_A + R_C - P = 0
eq_equilibrium = sym.Eq(R_A + R_C, P_load)

# 2. Compatibility Equation: delta_1 + delta_2 = 0
eq_compatibility = sym.Eq((R_A * L_1) / (A_1 * E) - (R_C * L_2) / (A_2 * E), 0)

# Solve simultaneous linear equations using sym.linsolve
reaction_soln, = sym.linsolve([eq_equilibrium, eq_compatibility], (R_A, R_C))

R_A_soln = reaction_soln[0].simplify()
R_C_soln = reaction_soln[1].simplify()

display(Math(r'R_A = {}'.format(sym.latex(R_A_soln))))
display(Math(r'R_C = {}'.format(sym.latex(R_C_soln))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 4. Direct Shear Stress and Factor of Safety (FOS)
A bolt of diameter $d$ connects two plates in a **double shear** configuration under tensile force $P$.
* Total shear area: $A_v = 2 \times \left(\frac{\pi d^2}{4}\right) = \frac{\pi d^2}{2}$
* Average shear stress: $\tau = \frac{P}{A_v}$

If the ultimate shear strength of the pin material is $\tau_{\text{ult}}$, the allowable working load is:
$$P_{\text{allowable}} = \frac{\tau_{\text{ult}} \cdot A_v}{\text{FOS}}$$

In [5]:
d, tau_ult, FOS = sym.symbols('d, tau_ult, FOS', positive=True)

# Shear area for double shear
A_shear = sym.pi * (d**2) / 2

# Allowable force
P_allowable = (tau_ult * A_shear) / FOS

display(Math(r'A_v = {}'.format(sym.latex(A_shear))))
display(Math(r'P_{{\text{{allowable}}}} = {}'.format(sym.latex(P_allowable))))

# Numerical test: d = 12 mm, tau_ult = 300 MPa, FOS = 2.5
P_num = P_allowable.subs({d: 12, tau_ult: 300, FOS: 2.5})
display(Math(r'P_{{\text{{allowable, num}}}} = {}\text{{ N}}'.format(sym.N(P_num, 5))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>